In [16]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import skew, kurtosis, norm, chi2

In [ ]:
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

def fmt(val, sig=""):
    return f"{val:.4f}{sig}"

def z_pval(z):
    return 2 * norm.sf(abs(z))

1.1 Consider the daily stock returns of American Express (AXP), Caterpillar(CAT), and Starbucks (SBUX) from January 1999 to December 2008. The data are simple returns given in the ﬁle `d-3stocks9908.txt` (date, axp, cat, sbux).

In [17]:
df = pd.read_csv("data/d-3stocks9908.txt", sep=r"\s+")
stocks = df.columns[1:]
df.head()

,date,axp,cat,sbux
0,19990104,-0.009756,0.029891,-0.040089
1,19990105,-0.019089,-0.002639,-0.034803
2,19990106,0.043063,0.026455,-0.008413
3,19990107,0.012063,0.009021,0.003636
4,19990108,0.030393,0.042146,0.021739


1. Express the simple returns in percentages. Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum, and maximum of the percentage simple returns.

In [18]:
ret_pct = df[stocks] * 100
pd.DataFrame({
    'mean': ret_pct.mean(),
    'std': ret_pct.std(ddof=1),
    'skewness': ret_pct.apply(skew),
    'excess_kurtosis': ret_pct.apply(lambda x: kurtosis(x, fisher=True)),
    'min': ret_pct.min(),
    'max': ret_pct.max()
})

,mean,std,skewness,excess_kurtosis,min,max
axp,0.014565,2.446218,-0.034627,6.055251,-17.5949,17.9266
cat,0.059504,2.169648,0.011678,4.459195,-14.5175,14.7229
sbux,0.048054,2.682622,-0.082476,8.754924,-28.2862,14.6354


2. Transform the simple returns to log returns.

In [19]:
log_ret_pct = np.log(df[stocks] + 1) * 100
log_ret_pct

,axp,cat,sbux
0,-0.980390,2.945297,-4.091471
1,-1.927355,-0.264249,-3.542305
2,4.216158,2.611112,-0.844859
3,1.199082,0.898055,0.362941
4,2.994028,4.128205,2.150608
...,...,...,...
2510,0.055684,1.878642,1.619023
2511,-0.334459,1.914261,0.107043
2512,-1.179428,-0.893480,-3.482439
2513,1.680697,3.069990,3.589307


3. Express the log returns in percentages. Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum, and maximum of the percentage log returns.

In [20]:
state = pd.DataFrame(
    {
        "mean": log_ret_pct.mean(),
        "std": log_ret_pct.std(ddof=0),
        "skewness": log_ret_pct.apply(skew),
        "excess_kurtosis": log_ret_pct.apply(lambda x: kurtosis(x, fisher=True)),
        "min": log_ret_pct.min(),
        "max": log_ret_pct.max(),
    }
)
state

,mean,std,skewness,excess_kurtosis,min,max
axp,-0.015434,2.452410,-0.336635,6.494046,-19.352286,16.489221
cat,0.035949,2.171051,-0.201865,4.700869,-15.685851,13.734947
sbux,0.011885,2.695352,-0.597424,12.908121,-33.248699,13.658647


4. Test the null hypothesis that the mean of the log returns of each stock is zero. That is, perform three separate tests. Use 5% signiﬁcance level to draw your conclusion.

In [21]:
print("Reject H0?: ")
critical = 1.96  # Z0.975
np.abs(state["mean"] / (state["std"] / len(df) ** 0.5)) > critical

Reject H0?: 


axp     False
cat     False
sbux    False
dtype: bool

1.4. Consider the daily log returns of American Express (AXP), Caterpillar (CAT), and Starbucks (SBUX) from January 1999 to December 2008 as in Exercise 1.1. Use the 5% significance level to perform the following tests for each stock:
1. Test H₀: mean = 0
2. Test H₀: skewness = 0
3. Test H₀: excess kurtosis = 0
4. Jarque-Bera test for normality

In [ ]:
tickers = ["QQQ", "MCD", "KO", "^GSPC"]
raw = yf.download(
    tickers,
    start="2016-03-17",
    end="2026-03-20",
    auto_adjust=True,
    progress=False,
)["Close"]
simple_ret = raw.pct_change().dropna()
log_ret_my = np.log(simple_ret + 1) * 100


def build_row(r):
    r = np.asarray(r, dtype=float)
    T = len(r)

    mu = r.mean()
    sigma = ((r - mu) ** 2).mean() ** 0.5

    sk = ((r - mu) ** 3).mean() / sigma**3
    ek = ((r - mu) ** 4).mean() / sigma**4 - 3

    jb = T / 6 * (sk**2 + ek**2 / 4)
    jb_p = chi2.sf(jb, df=2)

    p_mu = z_pval(mu / (sigma / T**0.5))
    p_sk = z_pval(sk / (6 / T) ** 0.5)
    p_ek = z_pval(ek / (24 / T) ** 0.5)

    return {
        "Mean": fmt(mu, stars(p_mu)),
        "Std Dev": fmt(sigma),
        "Skewness": fmt(sk, stars(p_sk)),
        "Excess Kurtosis": fmt(ek, stars(p_ek)),
        "Minimum": fmt(r.min()),
        "Maximum": fmt(r.max()),
        "Jarque-Bera": fmt(jb, stars(jb_p)),
    }


rows = {}
for col in ["axp", "cat", "sbux"]:
    rows[col.upper()] = build_row(log_ret_pct[col])

for ticker in tickers:
    rows[f"{ticker}"] = build_row(log_ret_my[ticker])

result = pd.DataFrame(rows).T

print(f"Original stocks: Daily Log Returns (%)  n = {len(log_ret_pct)}  [1999-2008]")
print(f"Custom stocks:   Daily Log Returns (%)  n = {len(log_ret_my)}  [2014-2024]")
print("Signif. codes:  *** 0.001  ** 0.01  * 0.05\n")
result

1.2. Answer the same questions as in Exercise 1.1 but using monthly stock returns for General Motors (GM), CRSP value-weighted index (VW), CRSP equalweighted index (EW), and S&P composite index from January 1975 to December 2008. The returns of the indexes include dividend distributions. Data ﬁle is `m-gm3dx7508.txt` (date, gm, vw, ew, sp).

In [23]:
df = pd.read_csv("data/m-gm3dx7508.txt", sep=r"\s+")
stocks = df.columns[1:]
df.head()

,date,gm,vw,ew,sp
0,19750131,0.252033,0.141600,0.299260,0.122812
1,19750228,0.028571,0.058411,0.053918,0.059886
2,19750331,0.054487,0.030191,0.081497,0.021694
3,19750430,0.045593,0.046497,0.031093,0.047265
4,19750530,0.037209,0.055140,0.072876,0.044101


1. Express the simple returns in percentages. Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum, and maximum of the percentage simple returns.

In [ ]:
ret_pct = df[stocks] * 100

def build_row_12(r):
    r = np.asarray(r, dtype=float)
    T = len(r)

    mu = r.mean()
    sigma = ((r - mu) ** 2).mean() ** 0.5
    sk = ((r - mu) ** 3).mean() / sigma**3
    ek = ((r - mu) ** 4).mean() / sigma**4 - 3

    jb = T / 6 * (sk**2 + ek**2 / 4)
    jb_p = chi2.sf(jb, df=2)
    p_mu = z_pval(mu / (sigma / T**0.5))
    p_sk = z_pval(sk / (6 / T) ** 0.5)
    p_ek = z_pval(ek / (24 / T) ** 0.5)

    return {
        "Mean": fmt(mu, stars(p_mu)),
        "Std Dev": fmt(sigma),
        "Skewness": fmt(sk, stars(p_sk)),
        "Excess Kurtosis": fmt(ek, stars(p_ek)),
        "Minimum": fmt(r.min()),
        "Maximum": fmt(r.max()),
        "Jarque-Bera": fmt(jb, stars(jb_p)),
    }

rows = {col.upper(): build_row_12(ret_pct[col]) for col in stocks}
result_simple = pd.DataFrame(rows).T

print(f"Monthly Simple Returns (%)  n = {len(ret_pct)}  [1975-2008]")
print("Signif. codes:  *** 0.001  ** 0.01  * 0.05\n")
result_simple

2. Transform the simple returns to log returns.

In [25]:
log_ret_pct = np.log(df[stocks] + 1) * 100
log_ret_pct

,gm,vw,ew,sp
0,22.476863,13.243079,26.179487,11.583625
1,2.817046,5.676873,5.251465,5.816136
2,5.305439,2.974422,7.834619,2.146203
3,4.458419,4.544840,3.061940,4.618200
4,3.653345,5.367346,7.034289,4.315623
...,...,...,...,...
403,-10.165406,1.098148,1.396799,1.211729
404,-5.657035,-10.320728,-12.884297,-9.518029
405,-48.988312,-20.423103,-23.067686,-18.563705
406,-9.981039,-8.905638,-14.687372,-7.779831


3. Express the log returns in percentages. Compute the sample mean, standard deviation, skewness, excess kurtosis, minimum, and maximum of the percentage log returns.

In [26]:
state = pd.DataFrame(
    {
        "mean": log_ret_pct.mean(),
        "std": log_ret_pct.std(ddof=0),
        "skewness": log_ret_pct.apply(skew),
        "excess_kurtosis": log_ret_pct.apply(lambda x: kurtosis(x, fisher=True)),
        "min": log_ret_pct.min(),
        "max": log_ret_pct.max(),
    }
)
state

,mean,std,skewness,excess_kurtosis,min,max
gm,0.110182,9.578516,-1.027439,4.055294,-49.317073,24.421518
vw,0.904567,4.554993,-1.054877,3.971681,-25.536075,13.243079
ew,1.166997,5.619083,-0.839217,5.283005,-31.779495,26.179487
sp,0.631937,4.396808,-0.857996,3.365860,-24.542750,12.378013


4. Test the null hypothesis that the mean of the log returns of each stock is zero. That is, perform three separate tests. Use 5% signiﬁcance level to draw your conclusion.

In [ ]:
rows = {col.upper(): build_row_12(log_ret_pct[col]) for col in stocks}
result_log = pd.DataFrame(rows).T

print(f"Monthly Log Returns (%)  n = {len(log_ret_pct)}  [1975-2008]")
print("Signif. codes:  *** 0.001  ** 0.01  * 0.05\n")
result_log

1.3. Consider the monthly stock returns of S&P composite index from January 1975 to December 2008 in Exercise 1.2. Answer the following questions:
1. What is the average annual log return over the data span?

In [28]:
sp_log_ret = log_ret_pct["sp"]

# Average annual log return: group into years (12 months), compute annual log return, then annualize
annual_log_ret = np.reshape(sp_log_ret.values, [-1, 12]).sum(axis=1)  # sum of monthly log returns = annual log return
annual_simple_ret = np.exp(annual_log_ret / 100) - 1
annual_simple_ret.mean()

np.float64(0.09307990920744352)

2. Assume that there were no transaction costs. If one invested $1.00 on the S&P composite index at the beginning of 1975, what was the value of the investment at the end of 2008?

In [29]:
np.exp(sp_log_ret.sum() / 100)

np.float64(13.1747742025844)